In [2]:
import numpy as np
import pandas as pd
import glob 
import os 
from pycocotools.coco import COCO

In [3]:
data = np.load(r'C:\Users\moham\Downloads\2026_geoai_arctic_challenge_v1.0\competition_release\train\images\train_000001.npz')
print(f"Arrays:{data}")
for key in data.files:
    arr = data[key]
    print(f"key :{key} , shape: {arr.shape}") 
print(len(data[key][0][0]))

Arrays:NpzFile 'C:\\Users\\moham\\Downloads\\2026_geoai_arctic_challenge_v1.0\\competition_release\\train\\images\\train_000001.npz' with keys: image
key :image , shape: (159, 289, 8)
8


In [4]:
annotation_path  = r"C:\Users\moham\Downloads\2026_geoai_arctic_challenge_v1.0\competition_release\train\annotations\instances_train.json"
coco = COCO(annotation_path)

filename_to_img_info = {img["file_name"]: img for img in coco.loadImgs(coco.getImgIds())}

npz_files = sorted(glob.glob(r"C:\Users\moham\Downloads\2026_geoai_arctic_challenge_v1.0\competition_release\train\images\*.npz"))


pixel_data_list = []

for npz_path in npz_files:
    file_name = os.path.basename(npz_path)
    file_name = f"images/{file_name}"



    npz_data = np.load(npz_path)

    img_array = npz_data["image"]

    H , W , C = img_array.shape

    mask = np.zeros((H,W) , dtype = np.int8)

    if file_name in filename_to_img_info:
        img_info = filename_to_img_info[file_name]
        ann_ids = coco.getAnnIds(imgIds=img_info['id'])
        anns = coco.loadAnns(ann_ids)

        for ann in anns:
            binary_mask = coco.annToMask(ann)
            mask[binary_mask == 1] = 1

    X_pixels = img_array.reshape(-1,C)
    y_pixels = mask.reshape(-1,1)

    img_pixel_table = np.hstack([X_pixels, y_pixels])
    pixel_data_list.append(img_pixel_table)

full_dataset = np.vstack(pixel_data_list)

channel_cols = [f"channel_{i+1}" for i in range(C)]
df = pd.DataFrame(full_dataset, columns=channel_cols + ["target"])

print(f"Total Rows (Pixels across all images): {df.shape[0]}")
print(f"Total Columns (Features + Target): {df.shape[1]}")


loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Total Rows (Pixels across all images): 38677917
Total Columns (Features + Target): 9


In [5]:
df.to_parquet("pixel_dataset.parquet", index=False, compression="snappy")

In [6]:
df = pd.read_parquet(r"C:\Users\moham\Downloads\2026_geoai_arctic_challenge_v1.0\Dataframe_prep\pixel_dataset.parquet")
df.head()

,channel_1,channel_2,channel_3,channel_4,channel_5,channel_6,channel_7,channel_8,target
0,190.0,175.0,152.0,0.146872,0.271973,148.420029,1417.0,-0.244772,0.0
1,190.0,178.0,154.0,0.090082,0.556396,148.874649,1226.0,-0.229881,0.0
2,181.0,169.0,145.0,0.090082,0.774902,150.379013,1033.0,-0.229881,0.0
3,177.0,167.0,142.0,0.090082,0.914062,153.140991,1033.0,-0.229881,0.0
4,176.0,164.0,140.0,0.090082,0.967041,158.649612,923.0,-0.229881,0.0


In [10]:
df[df["target"] == 1]

,channel_1,channel_2,channel_3,channel_4,channel_5,channel_6,channel_7,channel_8,target
48,36.0,40.0,43.0,0.071286,-0.393311,173.416077,1663.5,-0.171756,1.0
49,29.0,32.0,37.0,0.071286,-0.630127,173.262787,1663.0,-0.171756,1.0
50,57.0,58.0,62.0,0.071286,-0.838135,173.325134,1672.0,-0.171756,1.0
51,47.0,51.0,54.0,0.071286,-0.989502,173.232178,1693.0,-0.171756,1.0
52,59.0,60.0,62.0,0.071286,-1.065186,172.814240,1728.5,-0.171756,1.0
...,...,...,...,...,...,...,...,...,...
38677794,80.0,87.0,71.0,0.469703,-0.861816,151.755035,1932.0,-0.497701,1.0
38677795,82.0,89.0,73.0,0.469703,-0.523926,149.248596,1932.0,-0.497701,1.0
38677796,84.0,88.0,71.0,0.535966,-0.816895,145.877289,2087.0,-0.567470,1.0
38677797,84.0,88.0,71.0,0.535966,-0.706543,150.556046,2209.0,-0.567470,1.0
